# GeneScopeAI — Qwen3-1.7B Evidence Classification Fine-Tuning

This notebook reproduces a **single group-safe 96/24 train/validation experiment** comparing candidate-label scoring before and after LoRA fine-tuning.

**What you need:** Google Colab with a T4 GPU (or compatible CUDA GPU), `GeneScopeAI_Training_FINAL.csv` containing `id`, `claim`, `evidence`, `label`, `group_id`, and access to the public `Qwen/Qwen3-1.7B-Base` model.

**Run cells top-to-bottom** in a fresh Colab GPU session. When prompted by `files.upload()`, choose `GeneScopeAI_Training_FINAL.csv`. The notebook regenerates the group-safe split using a fixed seed; do **not** alter the split for the baseline-vs-fine-tuned comparison.

**Prior reported experiment:** baseline 22/24 (91.67%) and fine-tuned 24/24 (100%) on a small, synthetic, source-informed validation set. These scores are not guarantees of reproducibility or real-world performance. Evaluation examples are not used for gradient updates, but this set has been inspected during experimentation; it is **not** a final independent test set. Original source passage verification is tracked separately from classification review.

**Public repository note:** This copy has no saved cell outputs. It should not contain authentication tokens or private data; check any new edits before publishing.



In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Upload the final CSV


In [ ]:
from google.colab import files

uploaded = files.upload()


## 2. Verify the dataset


In [ ]:

import pandas as pd

training_df = pd.read_csv("GeneScopeAI_Training_FINAL.csv")

print("Total records:", len(training_df))

print("\nColumns:")
print(training_df.columns.tolist())

print("\nLabel distribution:")
print(training_df["label"].value_counts())

print("\nFirst 3 records:")
display(training_df.head(3))


## 3. Create a group-safe 80/20 split


In [ ]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Use the training dataset already loaded
data = training_df.copy()

# Check required columns
required = ["id", "claim", "evidence", "label", "group_id"]
assert data[required].notna().all().all(), "Missing required values"
assert data["id"].is_unique, "Duplicate record IDs found"

labels = [
    "supported",
    "contradicted",
    "insufficient_evidence"
]

assert set(data["label"]) == set(labels)

# Generate group-safe split candidates
splitter = GroupShuffleSplit(
    n_splits=300,
    test_size=0.20,
    random_state=42
)

best = None
best_score = float("inf")

for train_idx, val_idx in splitter.split(
    data,
    y=data["label"],
    groups=data["group_id"]
):
    train_candidate = data.iloc[train_idx]
    val_candidate = data.iloc[val_idx]

    # Require all three labels in both sets
    if set(train_candidate["label"]) != set(labels):
        continue
    if set(val_candidate["label"]) != set(labels):
        continue

    # Prefer ~20% validation and balanced labels
    size_error = abs(len(val_candidate) / len(data) - 0.20)
    overall = data["label"].value_counts(normalize=True)
    actual = val_candidate["label"].value_counts(normalize=True)
    balance_error = sum(abs(actual[l] - overall[l]) for l in labels)

    score = size_error + balance_error

    if score < best_score:
        best_score = score
        best = (train_idx, val_idx)

assert best is not None, "No suitable group-safe split found"

train_df = data.iloc[best[0]].copy()
val_df = data.iloc[best[1]].copy()

# Verify no shared concepts
overlap = set(train_df["group_id"]) & set(val_df["group_id"])
assert len(overlap) == 0, "Data leakage detected"

# Save both files
train_df.to_csv("GeneScopeAI_train.csv", index=False)
val_df.to_csv("GeneScopeAI_validation.csv", index=False)

print("Training records:", len(train_df))
print("Validation records:", len(val_df))
print("Shared group IDs:", len(overlap))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nFiles created successfully!")


In [ ]:

from google.colab import files

files.download("GeneScopeAI_train.csv")
files.download("GeneScopeAI_validation.csv")


## 4. Install baseline evaluation dependencies


In [ ]:

%pip install -q -U "transformers>=4.51.0,<5" accelerate bitsandbytes scikit-learn


In [ ]:

import torch
import transformers
import accelerate
import bitsandbytes
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("\nGPU available:", torch.cuda.is_available())
print("Library imports successful!")


## 5. Load and evaluate the original base model


In [ ]:

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# Original model for our experiment
MODEL_ID = "Qwen/Qwen3-1.7B-Base"

# Use 4-bit quantization to save GPU memory
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    dtype=torch.float16
)

# Set model to evaluation mode
model.eval()

print("\nModel loaded successfully!")
print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

print("Model loaded:", "model" in globals())

if "model" in globals():
    print("Model device:", model.device)
    print("Model name:", model.config._name_or_path)


In [ ]:

import torch
import torch.nn.functional as F

LABELS = [
    "supported",
    "contradicted",
    "insufficient_evidence"
]

def make_prompt(claim, evidence):
    return (
        "Classify the claim using ONLY the supplied evidence.\n"
        "Choose one label:\n"
        "supported: evidence establishes the claim.\n"
        "contradicted: evidence directly refutes the claim.\n"
        "insufficient_evidence: evidence does not establish "
        "or directly refute the claim.\n\n"
        f"Evidence: {evidence}\n"
        f"Claim: {claim}\n"
        "Label:"
    )

@torch.inference_mode()
def predict_label(claim, evidence):
    prompt = make_prompt(claim, evidence)

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    ).input_ids

    # Compare how likely the model considers each label
    scores = {}

    for label in LABELS:
        label_ids = tokenizer(
            " " + label,
            add_special_tokens=False
        ).input_ids

        ids = torch.tensor(
            [prompt_ids + label_ids],
            device="cuda"
        )

        outputs = model(input_ids=ids)
        logits = outputs.logits[0]

        start = len(prompt_ids) - 1
        end = start + len(label_ids)

        log_probs = F.log_softmax(
            logits[start:end].float(),
            dim=-1
        )

        target = torch.tensor(
            label_ids,
            device="cuda"
        )

        token_scores = log_probs.gather(
            1,
            target.unsqueeze(1)
        ).squeeze(1)

        # Normalize to compare labels of different lengths
        scores[label] = token_scores.mean().item()

    prediction = max(scores, key=scores.get)
    return prediction, scores

# Test on the first held-out example
example = val_df.iloc[0]

prediction, scores = predict_label(
    example["claim"],
    example["evidence"]
)

print("Record:", example["id"])
print("Actual label:", example["label"])
print("Qwen prediction:", prediction)
print("Correct:", prediction == example["label"])
print("\nLabel scores:")
print(scores)


In [ ]:

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Use the original Qwen model
model.eval()

results = []

# Evaluate all 24 held-out examples
for i, row in val_df.iterrows():

    prediction, scores = predict_label(
        row["claim"],
        row["evidence"]
    )

    results.append({
        "id": row["id"],
        "actual": row["label"],
        "predicted": prediction,
        "correct": row["label"] == prediction,
        "supported_score": scores["supported"],
        "contradicted_score": scores["contradicted"],
        "insufficient_evidence_score": scores["insufficient_evidence"]
    })

    print(
        f"{len(results)}/24 | "
        f"{row['id']} | "
        f"Actual: {row['label']} | "
        f"Predicted: {prediction}"
    )

baseline_df = pd.DataFrame(results)

# Save baseline predictions
baseline_df.to_csv(
    "GeneScopeAI_baseline_predictions.csv",
    index=False
)

# Calculate accuracy
accuracy = accuracy_score(
    baseline_df["actual"],
    baseline_df["predicted"]
)

print("\n" + "=" * 45)
print("BASELINE EVALUATION RESULTS")
print("=" * 45)

print(f"\nBaseline Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(
    classification_report(
        baseline_df["actual"],
        baseline_df["predicted"],
        labels=LABELS,
        zero_division=0
    )
)

# Create confusion matrix
cm = confusion_matrix(
    baseline_df["actual"],
    baseline_df["predicted"],
    labels=LABELS
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=LABELS
)

fig, ax = plt.subplots(figsize=(9, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=25)
ax.set_title("GeneScopeAI - Qwen Baseline Confusion Matrix")
plt.tight_layout()
plt.savefig(
    "GeneScopeAI_baseline_confusion_matrix.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()


In [ ]:

from google.colab import files

files.download("GeneScopeAI_baseline_predictions.csv")
files.download("GeneScopeAI_baseline_confusion_matrix.png")


In [ ]:

import pandas as pd

# We already created train_df and val_df
print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))

# Build a training example using the same prompt
# format as the baseline evaluation
def format_training_example(row):
    prompt = make_prompt(
        row["claim"],
        row["evidence"]
    )

    answer = " " + row["label"]

    return {
        "id": row["id"],
        "prompt": prompt,
        "completion": answer
    }

formatted_train = [
    format_training_example(row)
    for _, row in train_df.iterrows()
]

print("\nFirst formatted training example:")
print(formatted_train[0]["prompt"])
print(formatted_train[0]["completion"])

print("\nTotal formatted examples:", len(formatted_train))


## 6. Install PEFT and configure the LoRA adapter


In [ ]:

%pip install -q "peft>=0.17,<0.21"


In [ ]:

import peft
import torch

print("PEFT version:", peft.__version__)
print("GPU available:", torch.cuda.is_available())
print("Qwen loaded:", "model" in globals())
print("Training examples:", len(train_df))
print("Validation examples:", len(val_df))


In [ ]:

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# Prepare the existing 4-bit model for LoRA training
model = prepare_model_for_kbit_training(model)

# Configure the LoRA adapter
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

# Attach the adapter
model = get_peft_model(model, lora_config)

# Show how many parameters will be trained
model.print_trainable_parameters()


## 7. Prepare completion-only training data


In [ ]:

import torch

# Prepare Qwen's padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LENGTH = 512

def prepare_training_record(row):
    prompt = make_prompt(
        row["claim"],
        row["evidence"]
    )

    answer = " " + row["label"] + tokenizer.eos_token

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False
    )["input_ids"]

    answer_ids = tokenizer(
        answer,
        add_special_tokens=False
    )["input_ids"]

    # Preserve the answer when applying the length limit
    max_prompt_length = MAX_LENGTH - len(answer_ids)

    if max_prompt_length <= 0:
        raise ValueError("Answer exceeds maximum length")

    prompt_ids = prompt_ids[:max_prompt_length]

    input_ids = prompt_ids + answer_ids

    # -100 tells PyTorch to ignore the prompt in the loss
    labels = (
        [-100] * len(prompt_ids)
        + answer_ids
    )

    return {
        "input_ids": torch.tensor(
            input_ids, dtype=torch.long
        ),
        "attention_mask": torch.ones(
            len(input_ids), dtype=torch.long
        ),
        "labels": torch.tensor(
            labels, dtype=torch.long
        )
    }

training_records = [
    prepare_training_record(row)
    for _, row in train_df.iterrows()
]

print("Prepared training records:", len(training_records))
print("First record token count:",
      len(training_records[0]["input_ids"]))

print("Prompt tokens excluded from loss:",
      (training_records[0]["labels"] == -100).sum().item())

print("Answer tokens included in loss:",
      (training_records[0]["labels"] != -100).sum().item())

print("\nTraining preparation successful!")


## 8. Train and save the LoRA adapter


In [ ]:

import os
import math
import torch

from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import get_linear_schedule_with_warmup

# Reproducibility
torch.manual_seed(42)

# Training settings
EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 1e-4
OUTPUT_DIR = "GeneScopeAI_LoRA_Adapter"

# Pad examples to the same length within each batch
def collate_batch(batch):
    input_ids = pad_sequence(
        [x["input_ids"] for x in batch],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    attention_mask = pad_sequence(
        [x["attention_mask"] for x in batch],
        batch_first=True,
        padding_value=0
    )

    labels = pad_sequence(
        [x["labels"] for x in batch],
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_loader = DataLoader(
    training_records,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

# Optimize only the LoRA parameters
trainable_params = [
    p for p in model.parameters() if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.01
)

steps_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION
)

total_steps = EPOCHS * steps_per_epoch

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(total_steps * 0.1)),
    num_training_steps=total_steps
)

# Train
model.config.use_cache = False
model.train()
optimizer.zero_grad(set_to_none=True)

loss_history = []

print("Starting GeneScopeAI LoRA fine-tuning")
print("Training records:", len(training_records))
print("Epochs:", EPOCHS)
print("Optimizer steps:", total_steps)

for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for step, batch in enumerate(train_loader):
        batch = {
            k: v.to("cuda") for k, v in batch.items()
        }

        outputs = model(**batch)
        loss = outputs.loss

        # Track original loss before scaling
        epoch_loss += loss.item()

        # Gradient accumulation
        (loss / GRADIENT_ACCUMULATION).backward()

        last_batch = (step + 1 == len(train_loader))

        if ((step + 1) % GRADIENT_ACCUMULATION == 0
                or last_batch):

            torch.nn.utils.clip_grad_norm_(
                trainable_params, max_norm=1.0
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"| Average training loss: {avg_loss:.4f}"
    )

# Save the trained LoRA adapter
model.eval()
model.config.use_cache = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("\nTraining completed successfully!")
print("Adapter saved to:", OUTPUT_DIR)
print("Training losses:", loss_history)


In [ ]:

import shutil
from google.colab import files

shutil.make_archive(
    "GeneScopeAI_LoRA_Adapter",
    "zip",
    "GeneScopeAI_LoRA_Adapter"
)

files.download("GeneScopeAI_LoRA_Adapter.zip")


## 9. Re-evaluate the adapter using the same validation examples


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import pandas as pd
import matplotlib.pyplot as plt

# Use the fine-tuned model
model.eval()

fine_tuned_results = []

for _, row in val_df.iterrows():

    prediction, scores = predict_label(
        row["claim"],
        row["evidence"]
    )

    fine_tuned_results.append({
        "id": row["id"],
        "actual": row["label"],
        "predicted": prediction,
        "correct": prediction == row["label"]
    })

    print(
        f"{len(fine_tuned_results)}/24 | "
        f"{row['id']} | "
        f"Actual: {row['label']} | "
        f"Predicted: {prediction}"
    )

fine_tuned_df = pd.DataFrame(fine_tuned_results)

# Calculate accuracy
fine_tuned_accuracy = accuracy_score(
    fine_tuned_df["actual"],
    fine_tuned_df["predicted"]
)

print("\n==================================")
print("FINE-TUNED MODEL RESULTS")
print("==================================")

print(
    f"Fine-tuned accuracy: "
    f"{fine_tuned_accuracy:.2%}"
)

print("\nClassification Report:")

print(
    classification_report(
        fine_tuned_df["actual"],
        fine_tuned_df["predicted"],
        labels=LABELS,
        zero_division=0
    )
)

# Create confusion matrix
cm = confusion_matrix(
    fine_tuned_df["actual"],
    fine_tuned_df["predicted"],
    labels=LABELS
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=LABELS
)

fig, ax = plt.subplots(figsize=(9, 6))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=25)
ax.set_title("GeneScopeAI - Fine-Tuned Qwen")

plt.tight_layout()
plt.savefig(
    "GeneScopeAI_finetuned_confusion_matrix.png",
    dpi=160,
    bbox_inches="tight"
)
plt.show()

# Save predictions
fine_tuned_df.to_csv(
    "GeneScopeAI_finetuned_predictions.csv",
    index=False
)

print("\nFine-tuned predictions saved!")


## 10. Compare predictions and export evidence


In [ ]:

comparison = baseline_df[
    ["id", "actual", "predicted"]
].rename(columns={
    "predicted": "baseline_prediction"
})

comparison = comparison.merge(
    fine_tuned_df[["id", "predicted"]].rename(
        columns={"predicted": "finetuned_prediction"}
    ),
    on="id",
    validate="one_to_one"
)

comparison["baseline_correct"] = (
    comparison["actual"] ==
    comparison["baseline_prediction"]
)

comparison["finetuned_correct"] = (
    comparison["actual"] ==
    comparison["finetuned_prediction"]
)

print("Baseline accuracy: 91.67%")
print(f"Fine-tuned accuracy: {fine_tuned_accuracy:.2%}")

print("\nPredictions that changed:")
display(
    comparison[
        comparison["baseline_prediction"] !=
        comparison["finetuned_prediction"]
    ]
)

comparison.to_csv(
    "GeneScopeAI_before_after_comparison.csv",
    index=False
)


In [ ]:

import json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from google.colab import files

def calculate_metrics(df):
    return {
        "accuracy": accuracy_score(df["actual"], df["predicted"]),
        "macro_precision": precision_score(
            df["actual"], df["predicted"],
            labels=LABELS, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            df["actual"], df["predicted"],
            labels=LABELS, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            df["actual"], df["predicted"],
            labels=LABELS, average="macro", zero_division=0
        )
    }

summary = {
    "project": "GeneScopeAI Evidence Classification",
    "base_model": "Qwen/Qwen3-1.7B-Base",
    "method": "4-bit LoRA (QLoRA)",
    "training_examples": 96,
    "validation_examples": 24,
    "epochs": 3,
    "optimizer_steps": 36,
    "shared_group_ids": 0,
    "baseline": calculate_metrics(baseline_df),
    "fine_tuned": calculate_metrics(fine_tuned_df),
    "limitations": [
        "Small validation sample of 24 records",
        "Synthetic source-informed dataset",
        "Not an independent external test set"
    ]
}

with open("GeneScopeAI_experiment_results.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

files.download("GeneScopeAI_experiment_results.json")
files.download("GeneScopeAI_before_after_comparison.csv")
